# Credit Risk Analysis using Machine Learning

End-to-end notebook: data preprocessing, EDA, feature engineering, model training (Logistic Regression, Decision Tree, Random Forest, XGBoost), and evaluation.

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from generate_data import generate_dataset
from data_preprocessing import load_data, engineer_features, encode_and_scale, get_train_test_split

sns.set_theme(style='whitegrid')

## 1. Load / Generate Dataset

In [ ]:
df = load_data()
df.head()

In [ ]:
df.info()
df.describe()

## 2. Exploratory Data Analysis

In [ ]:
df['default'].value_counts(normalize=True)

In [ ]:
numeric_df = df.select_dtypes(include=['int64','float64'])
plt.figure(figsize=(10,8))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 3. Feature Engineering, Encoding & Normalization

In [ ]:
df_fe = engineer_features(df)
X, y, scaler, encoders, feature_cols = encode_and_scale(df_fe)
X.head()

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test, scaler, encoders, feature_cols = get_train_test_split()
X_train.shape, X_test.shape

## 5. Model Training & Comparison
Logistic Regression, Decision Tree, Random Forest, XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
}
if HAS_XGB:
    models['XGBoost'] = XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.05, eval_metric='logloss', random_state=42)

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'precision': precision_score(y_test, pred),
        'recall': recall_score(y_test, pred),
        'f1_score': f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results).sort_values('f1_score', ascending=False)
results_df

## 6. Visualize Model Comparison

In [ ]:
results_df.set_index('model')[['accuracy','precision','recall','f1_score']].plot(kind='bar', figsize=(10,5))
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.show()